## Phase 1: Preprocessing & Sampling
#### File: song_wrangling.ipynb

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv('spotify.csv')
df.head(3)

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,...,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,...,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,...,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


In [3]:
# cleaning - removed unnecessary 'unnamed' id col, remove songs that have null artist, title, or album
# deduplicating - remove duplicate track_ids (remasters, different album listings)
# renaming - rename 'artists' col to 'artist' for neo4j property naming

df = df.drop(columns=['Unnamed: 0'])
df = df.dropna(subset=['track_name','artists','track_genre'])
df = df.drop_duplicates(subset='track_id', keep='first')
df = df.rename(columns={'artists':'artist'})

In [4]:
print(f'Total unique tracks after dedup: {len(df)}')
df.head(3)

Total unique tracks after dedup: 89740


,track_id,artist,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


In [5]:
# must haves
strokes_songs = df[df['artist'].str.contains('The Strokes', case=False, na=False)]
regina_songs = df[df['artist'].str.contains('Regina Spektor', case=False, na=False)]

print(f'The Strokes songs found: {len(strokes_songs)}')
print(f'Regina Spektor songs found: {len(regina_songs)}')

The Strokes songs found: 43
Regina Spektor songs found: 3


In [6]:
# stratify sample by genre - 20 songs per genre to ensure sonic diversity, enabling cross-genre recommendations
pool = df[~df['artist'].str.contains('The Strokes', case=False, na=False) &
          ~df['artist'].str.contains('Regina Spektor', case=False, na=False)]

genre_sample = pd.concat([
    group.sample(min(len(group), 20), random_state=22)
    for _, group in pool.groupby('track_genre')
]).reset_index(drop=True)

print(f'Stratified genre sample size: {len(genre_sample)}')

Stratified genre sample size: 2260


In [7]:
# add 200 of the most popular songs that were not found in the stratification above, ensuring that these popular songs (which have a high chance of being liked) are in the sample 
already_sampled_ids = set(genre_sample['track_id']) | set(strokes_songs['track_id']) | set(regina_songs['track_id'])
top_popular = (
    pool[~pool['track_id'].isin(already_sampled_ids)]
    .sort_values('popularity', ascending=False)
    .head(200)
)

In [8]:
sample = pd.concat([strokes_songs, regina_songs, genre_sample, top_popular])
sample = sample.drop_duplicates(subset='track_id', keep='first')

print(f'Final sample size: {len(sample)}')

Final sample size: 2506


In [ ]:
# normalize numerical sonic features - Min-Max normalization so that all features are on equal footing before weighting
SONIC_FEATURES = [
    'energy', 'valence', 'danceability', 'acousticness',
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

scaler = MinMaxScaler()
sample = sample.copy()
sample[SONIC_FEATURES] = scaler.fit_transform(sample[SONIC_FEATURES])

In [10]:
sample.head(3)

,track_id,artist,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
2206,5ruzrDWcT0vuJIOMW7gMnW,The Strokes,The New Abnormal,The Adults Are Talking,79,309053,False,0.612603,0.748993,5,0.832783,1,0.049376,0.011357,0.10600,0.302828,0.660224,0.766720,4,alt-rock
2463,3Y4rUyw7XBCK6hGHCOt6rp,The Strokes,Comedown Machine,"Call It Fate, Call It Karma",72,204773,False,0.561983,0.240979,4,0.579888,0,0.031185,0.987940,0.77400,0.083805,0.365209,0.508657,4,alt-rock
2554,2t0wwvR15fc3K1ey8OiOaN,The Strokes,The New Abnormal,Selfless,71,222093,False,0.555785,0.678991,4,0.870901,1,0.034615,0.191960,0.00175,0.071362,0.090336,0.567002,4,alt-rock


In [11]:
sample.to_csv('spotify_sample.csv', index=False)
print("Saved spotify_sample.csv")

Saved spotify_sample.csv


In [12]:
sample[['track_name','artist','track_genre']].sample(10)

,track_name,artist,track_genre
38371,Tap Out,The Strokes,garage
765,Savour The Flavour,Skegss,garage
17,Shallow,Boyce Avenue;Jennel Garcia,acoustic
1622,"Ek Din Aap (From ""Yes Boss"")",Kumar Sanu;Alka Yagnik,pop-film
2010,Soy Soledad,Inspector;Lng Sht,ska
1384,Yellow,Aromal Chekaver,malay
2763,Reptilia,The Strokes,alt-rock
67602,Te Felicito,Shakira;Rauw Alejandro,latin
4,Let It Go,Boyce Avenue;Rachel Grae,acoustic
1429,Happy Song,Bring Me The Horizon,metal
